In [1]:
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

from math import log, exp
from scipy.stats import poisson
from scipy.stats import gamma
import numpy as np
from nba_api.stats.endpoints import leaguedashteamstats

In [2]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

In [3]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        elif category == 'player_blocks_steals':
            hits = (last_n_games['BLK'] + last_n_games['STL'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'L-{window}'] = round(hits / window, 2)


    return results

# Bayesian posterior prediction.

## Points

In [4]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]
res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_off_rtg = league_df['OFF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()

def compute_bayesian_lambda(player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
                           team_stats, league_avg_off_rtg, league_avg_def_rtg, 
                           league_avg_pace, home_flag, current_date, player_name, projectedStartingFive):
    """
    Compute Bayesian posterior lambda for Poisson model.
    
    Uses Gamma prior (conjugate to Poisson) with:
    - Prior mean based on historical performance and contextual factors
    - Prior strength based on sample size and confidence
    - Updates with recent observations
    """
    eplison = 1e-6
    
    # Get baseline from season data
    if len(player_df) < 5:
        baseline_mean = player_df_25['PTS'].mean() if not player_df_25.empty else 10.0
        baseline_games = len(player_df_25)
    else:
        baseline_mean = player_df['PTS'].mean()
        baseline_games = len(player_df)
    
    if baseline_mean <= 0:
        return None
    
    # Get team and opponent stats
    team_or = team_stats.at[player_team, 'OFF_RATING']
    team_pace = team_stats.at[player_team, 'PACE']
    opp_dr = team_stats.at[opp_team_id, 'DEF_RATING']
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    
    # Contextual adjustments for prior mean
    # Team offensive strength
    team_or_factor = team_or / league_avg_off_rtg
    
    # Opponent defensive weakness (lower def rating = easier matchup)
    opp_dr_factor = league_avg_def_rtg / opp_dr
    
    # Pace adjustment
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = expected_pace / league_avg_pace
    
    # Home court advantage
    home_factor = 1.03 if home_flag else 0.97
    
    # Days rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days
    if days_rested == 1:
        rest_factor = 0.93
    elif days_rested == 2:
        rest_factor = 1.00
    elif 3 <= days_rested <= 5:
        rest_factor = 1.02
    else:
        rest_factor = 0.98
    
    # Starting status adjustment
    # Starting players typically get more minutes and usage, leading to higher scoring
    is_starting = player_name in projectedStartingFive.get(player_team_abbr, [])
    if is_starting:
        # Starting players get a boost (typically 10-20% more minutes/usage)
        starting_factor = 1.12
    else:
        # Bench players typically score less (15-25% reduction)
        starting_factor = 0.85
    
    # Head-to-head adjustment
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty and not player_df_25.empty:
        h2h = player_df_25[player_df_25['OPP_ABBREVIATION'] == opp_team]
    h2h_factor = 1.0
    if not h2h.empty:
        h2h_avg = h2h['PTS'].mean()
        h2h_factor = h2h_avg / baseline_mean if baseline_mean > 0 else 1.0
        # Cap H2H factor to prevent extreme values
        h2h_factor = max(0.85, min(1.15, h2h_factor))
    
    # Prior mean: baseline adjusted by contextual factors
    prior_mean = baseline_mean * team_or_factor * opp_dr_factor * pace_factor * home_factor * rest_factor * h2h_factor * starting_factor
    
    # Prior strength: how much weight we give to the prior
    # More games = stronger prior, but also consider recency
    prior_strength = min(baseline_games, 40)  # Cap at 40 games for stability
    prior_strength = max(prior_strength, 5)   # Minimum 5 games
    
    # Gamma prior parameters
    # For Gamma(alpha, beta), mean = alpha/beta
    # We want mean = prior_mean, so alpha = prior_mean * beta
    # beta controls the strength (higher beta = tighter prior)
    prior_beta = prior_strength / 10.0  # Scale: 10 games = beta of 1
    prior_alpha = prior_mean * prior_beta
    
    # Ensure prior_alpha >= 1 for valid Gamma distribution
    if prior_alpha < 1:
        prior_alpha = 1.0
        prior_beta = prior_alpha / prior_mean
    
    # Get recent observations for likelihood
    # Use last 7 games, or all available if fewer
    recent_games = min(7, len(player_df))
    recent_pts = player_df['PTS'].tail(recent_games).values
    
    # Bayesian update: Gamma is conjugate to Poisson
    # Posterior: Gamma(alpha_0 + sum(x_i), beta_0 + n)
    posterior_alpha = prior_alpha + recent_pts.sum()
    posterior_beta = prior_beta + recent_games
    
    # Posterior mean (this is our adjusted lambda)
    lambda_adjusted = posterior_alpha / posterior_beta
    
    # Also get posterior variance for uncertainty quantification
    posterior_variance = posterior_alpha / (posterior_beta ** 2)
    posterior_std = np.sqrt(posterior_variance)
    
    return lambda_adjusted, posterior_std

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_pts = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_off_rtg, league_avg_def_rtg,
        league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_pts % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_pts) + 1)
    elif target_pts % 1 == 0:
        prob_over_poisson = poisson.sf(target_pts, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_pts), int(target_pts) + 1)
    else:
        # Handle other cases
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_pts,
        'L-5': round(count_line_hits(player_df, target_pts, 'player_points', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_pts, 'player_points', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_pts, 'player_points', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })

point_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
point_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS/player_points.csv', index=False)
point_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Jaden Ivey,8.5,0.2,0.1,0.07,0.958,0.042
1,Tre Jones,8.5,0.8,0.7,0.67,0.862,0.138
2,Bennedict Mathurin,21.5,0.8,0.4,0.27,0.839,0.161
3,Sandro Mamukelashvili,8.5,0.6,0.7,0.67,0.823,0.177
4,Andrew Nembhard,16.5,0.6,0.5,0.33,0.807,0.193
5,Keegan Murray,12.5,0.2,0.1,0.07,0.790,0.210
6,Cooper Flagg,15.5,0.6,0.6,0.53,0.776,0.224
7,Dillon Brooks,19.5,0.8,0.5,0.40,0.766,0.234
8,Tobias Harris,11.5,0.6,0.3,0.20,0.763,0.237
9,Klay Thompson,10.5,0.6,0.6,0.47,0.757,0.243


## Assists

In [5]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_assists')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

def compute_bayesian_lambda_assists(player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
                                   team_stats, league_avg_def_rtg, league_avg_pace,
                                   league_avg_ast_ratio, league_avg_tov, home_flag, current_date, player_name, projectedStartingFive):
    """
    Compute Bayesian posterior lambda for assists using Poisson model.
    
    Uses Gamma prior (conjugate to Poisson) with assists-specific contextual factors.
    """
    eplison = 1e-6
    
    # Get baseline from season data
    if len(player_df) < 5:
        baseline_mean = player_df_25['AST'].mean() if not player_df_25.empty else 3.0
        baseline_games = len(player_df_25)
    else:
        baseline_mean = player_df['AST'].mean()
        baseline_games = len(player_df)
    
    if baseline_mean <= 0:
        return None
    
    # Get team and opponent stats
    team_pace = team_stats.at[player_team, 'PACE']
    team_ast_ratio = team_stats.at[player_team, 'AST_RATIO']
    opp_dr = team_stats.at[opp_team_id, 'DEF_RATING']
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    opp_tov = team_stats.at[opp_team_id, 'TM_TOV_PCT']
    
    # Contextual adjustments for prior mean (assists-specific)
    # Team assist culture (pass-heavy teams create more assists)
    team_ast_ratio_factor = team_ast_ratio / league_avg_ast_ratio
    
    # Opponent defensive weakness (weaker defense = easier passes)
    opp_dr_factor = league_avg_def_rtg / opp_dr
    
    # Opponent turnover pressure (teams that force turnovers limit assists)
    opp_tov_factor = league_avg_tov / opp_tov
    
    # Pace adjustment (more possessions = more assist opportunities)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = expected_pace / league_avg_pace
    
    # Home court advantage (smaller effect for assists)
    home_factor = 1.02 if home_flag else 0.98
    
    # Days rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days
    if days_rested == 1:
        rest_factor = 0.93
    elif days_rested == 2:
        rest_factor = 1.00
    elif 3 <= days_rested <= 5:
        rest_factor = 1.02
    else:
        rest_factor = 0.98
    
    # Starting status adjustment
    # Starting players typically handle the ball more and play with better teammates
    is_starting = player_name in projectedStartingFive.get(player_team_abbr, [])
    if is_starting:
        # Starting players get more assists (typically 8-15% more)
        starting_factor = 1.10
    else:
        # Bench players typically get fewer assists
        starting_factor = 0.88
    
    # Head-to-head adjustment
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty and not player_df_25.empty:
        h2h = player_df_25[player_df_25['OPP_ABBREVIATION'] == opp_team]
    h2h_factor = 1.0
    if not h2h.empty:
        h2h_avg = h2h['AST'].mean()
        h2h_factor = h2h_avg / baseline_mean if baseline_mean > 0 else 1.0
        h2h_factor = max(0.85, min(1.15, h2h_factor))
    
    # Prior mean: baseline adjusted by contextual factors
    prior_mean = (baseline_mean * team_ast_ratio_factor * opp_dr_factor * 
                  opp_tov_factor * pace_factor * home_factor * rest_factor * h2h_factor * starting_factor)
    
    # Prior strength: how much weight we give to the prior
    prior_strength = min(baseline_games, 40)
    prior_strength = max(prior_strength, 5)
    
    # Gamma prior parameters
    prior_beta = prior_strength / 10.0
    prior_alpha = prior_mean * prior_beta
    
    # Ensure valid Gamma distribution
    if prior_alpha < 1:
        prior_alpha = 1.0
        prior_beta = prior_alpha / prior_mean
    
    # Get recent observations for likelihood (last 7 games)
    recent_games = min(7, len(player_df))
    recent_ast = player_df['AST'].tail(recent_games).values
    
    # Bayesian update: Gamma is conjugate to Poisson
    # Posterior: Gamma(alpha_0 + sum(x_i), beta_0 + n)
    posterior_alpha = prior_alpha + recent_ast.sum()
    posterior_beta = prior_beta + recent_games
    
    # Posterior mean (adjusted lambda)
    lambda_adjusted = posterior_alpha / posterior_beta
    
    # Posterior variance for uncertainty
    posterior_variance = posterior_alpha / (posterior_beta ** 2)
    posterior_std = np.sqrt(posterior_variance)
    
    return lambda_adjusted, posterior_std

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_ast = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_assists(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_def_rtg, league_avg_pace,
        league_avg_ast_ratio, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_ast % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_ast) + 1)
    elif target_ast % 1 == 0:
        prob_over_poisson = poisson.sf(target_ast, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_ast), int(target_ast) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_ast,
        'L-5': round(count_line_hits(player_df, target_ast, 'player_assists', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_ast, 'player_assists', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_ast, 'player_assists', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
assist_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
assist_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS/player_assists.csv', index=False)
assist_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Russell Westbrook,6.5,0.6,0.6,0.47,0.703,0.297
1,Jalen Brunson,6.5,0.6,0.6,0.47,0.691,0.309
2,Duncan Robinson,1.5,0.8,0.6,0.53,0.679,0.321
3,Pelle Larsson,3.5,0.8,0.7,0.53,0.646,0.354
4,Kyle Filipowski,1.5,0.8,0.8,0.73,0.632,0.368
5,Isaiah Collier,5.5,0.6,0.5,0.33,0.604,0.396
6,Davion Mitchell,6.5,0.4,0.5,0.53,0.604,0.396
7,Terance Mann,3.5,0.6,0.7,0.60,0.603,0.397
8,Jamal Shead,5.5,0.6,0.5,0.40,0.597,0.403
9,Coby White,5.0,0.6,0.3,0.20,0.568,0.432


# REBOUNDS

In [6]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_rebounds')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
league_avg_dreb = league_df['DREB_PCT'].mean()
league_avg_reb = league_df['REB_PCT'].mean()

def compute_bayesian_lambda_rebounds(player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
                                     team_stats, league_avg_pace, league_avg_reb,
                                     league_avg_oreb, league_avg_dreb, home_flag, current_date, player_name, projectedStartingFive):
    """
    Compute Bayesian posterior lambda for rebounds using Poisson model.
    
    Uses Gamma prior (conjugate to Poisson) with rebounds-specific contextual factors.
    """
    eplison = 1e-6
    
    # Get baseline from season data
    if len(player_df) < 5:
        baseline_mean = player_df_25['REB'].mean() if not player_df_25.empty else 5.0
        baseline_games = len(player_df_25)
    else:
        baseline_mean = player_df['REB'].mean()
        baseline_games = len(player_df)
    
    if baseline_mean <= 0:
        return None
    
    # Get team and opponent stats
    team_pace = team_stats.at[player_team, 'PACE']
    team_reb = team_stats.at[player_team, 'REB_PCT']
    team_oreb = team_stats.at[player_team, 'OREB_PCT']
    team_dreb = team_stats.at[player_team, 'DREB_PCT']
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    opp_reb = team_stats.at[opp_team_id, 'REB_PCT']
    opp_oreb = team_stats.at[opp_team_id, 'OREB_PCT']
    opp_dreb = team_stats.at[opp_team_id, 'DREB_PCT']
    
    # Contextual adjustments for prior mean (rebounds-specific)
    # Team rebounding culture (rebounding-focused teams)
    team_reb_factor = team_reb / league_avg_reb
    
    # Opponent gives up offensive rebounds (weak DREB% = more offensive boards available)
    opp_dreb_factor = league_avg_dreb / opp_dreb
    
    # Opponent gives up defensive rebounds (weak OREB% = more defensive boards available)
    opp_oreb_factor = league_avg_oreb / opp_oreb
    
    # Pace adjustment (more possessions = more missed shots = more rebounds)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = expected_pace / league_avg_pace
    
    # Home court advantage (minimal effect for rebounds)
    home_factor = 1.01 if home_flag else 0.99
    
    # Days rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days
    if days_rested == 1:
        rest_factor = 0.93
    elif days_rested == 2:
        rest_factor = 1.00
    elif 3 <= days_rested <= 5:
        rest_factor = 1.02
    else:
        rest_factor = 0.98
    
    # Starting status adjustment
    # Starting players typically get more rebounds due to more minutes and often being bigger/stronger
    is_starting = player_name in projectedStartingFive.get(player_team_abbr, [])
    if is_starting:
        # Starting players get more rebounds (typically 8-15% more)
        starting_factor = 1.10
    else:
        # Bench players typically get fewer rebounds
        starting_factor = 0.88
    
    # Head-to-head adjustment
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty and not player_df_25.empty:
        h2h = player_df_25[player_df_25['OPP_ABBREVIATION'] == opp_team]
    h2h_factor = 1.0
    if not h2h.empty:
        h2h_avg = h2h['REB'].mean()
        h2h_factor = h2h_avg / baseline_mean if baseline_mean > 0 else 1.0
        h2h_factor = max(0.85, min(1.15, h2h_factor))
    
    # Prior mean: baseline adjusted by contextual factors
    prior_mean = (baseline_mean * team_reb_factor * opp_dreb_factor * 
                  opp_oreb_factor * pace_factor * home_factor * rest_factor * h2h_factor * starting_factor)
    
    # Prior strength: how much weight we give to the prior
    prior_strength = min(baseline_games, 40)
    prior_strength = max(prior_strength, 5)
    
    # Gamma prior parameters
    prior_beta = prior_strength / 10.0
    prior_alpha = prior_mean * prior_beta
    
    # Ensure valid Gamma distribution
    if prior_alpha < 1:
        prior_alpha = 1.0
        prior_beta = prior_alpha / prior_mean
    
    # Get recent observations for likelihood (last 7 games)
    recent_games = min(7, len(player_df))
    recent_reb = player_df['REB'].tail(recent_games).values
    
    # Bayesian update: Gamma is conjugate to Poisson
    # Posterior: Gamma(alpha_0 + sum(x_i), beta_0 + n)
    posterior_alpha = prior_alpha + recent_reb.sum()
    posterior_beta = prior_beta + recent_games
    
    # Posterior mean (adjusted lambda)
    lambda_adjusted = posterior_alpha / posterior_beta
    
    # Posterior variance for uncertainty
    posterior_variance = posterior_alpha / (posterior_beta ** 2)
    posterior_std = np.sqrt(posterior_variance)
    
    return lambda_adjusted, posterior_std

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_reb = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_rebounds(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_reb,
        league_avg_oreb, league_avg_dreb, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_reb % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_reb) + 1)
    elif target_reb % 1 == 0:
        prob_over_poisson = poisson.sf(target_reb, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_reb), int(target_reb) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_reb,
        'L-5': round(count_line_hits(player_df, target_reb, 'player_rebounds', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_reb, 'player_rebounds', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_reb, 'player_rebounds', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
rebound_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
rebound_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS/player_rebounds.csv', index=False)
rebound_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Donovan Mitchell,4.5,0.8,0.8,0.53,0.837,0.163
1,Saddiq Bey,4.5,0.6,0.6,0.40,0.723,0.277
2,Kel'el Ware,11.0,0.8,0.8,0.53,0.699,0.301
3,Keegan Murray,5.5,0.0,0.0,0.00,0.650,0.350
4,Keyonte George,3.5,0.6,0.7,0.60,0.641,0.359
5,Zach Edey,8.5,0.4,0.2,0.13,0.623,0.377
6,Cedric Coward,5.0,0.6,0.7,0.60,0.599,0.401
7,Russell Westbrook,6.5,0.8,0.7,0.60,0.591,0.409
8,Donovan Clingan,10.5,0.6,0.3,0.33,0.587,0.413
9,Matas Buzelis,5.0,0.4,0.4,0.33,0.585,0.415


## Blocks

In [7]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_blocks')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()

def compute_bayesian_lambda_blocks(player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
                                   team_stats, league_avg_pace, home_flag, current_date, player_name, projectedStartingFive):
    """
    Compute Bayesian posterior lambda for blocks using Poisson model.
    
    Uses Gamma prior (conjugate to Poisson) with blocks-specific contextual factors.
    Blocks are rare events, so we use a more conservative prior.
    """
    eplison = 1e-6
    
    # Get baseline from season data
    if len(player_df) < 5:
        baseline_mean = player_df_25['BLK'].mean() if not player_df_25.empty else 0.5
        baseline_games = len(player_df_25)
    else:
        baseline_mean = player_df['BLK'].mean()
        baseline_games = len(player_df)
    
    if baseline_mean <= 0:
        baseline_mean = 0.1  # Minimum for blocks (very rare events)
    
    # Get team and opponent stats
    team_pace = team_stats.at[player_team, 'PACE']
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    
    # Contextual adjustments for prior mean (blocks-specific)
    # Pace adjustment (more possessions = more shot attempts = more block opportunities)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = expected_pace / league_avg_pace
    
    # Home court advantage (minimal effect for blocks)
    home_factor = 1.00  # No home/away effect for blocks
    
    # Days rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days
    if days_rested == 1:
        rest_factor = 0.93
    elif days_rested == 2:
        rest_factor = 1.00
    elif 3 <= days_rested <= 5:
        rest_factor = 1.02
    else:
        rest_factor = 0.98
    
    # Starting status adjustment
    # Starting players (especially big men) typically get more blocks due to more minutes and better positioning
    is_starting = player_name in projectedStartingFive.get(player_team_abbr, [])
    if is_starting:
        # Starting players get more blocks (typically 10-20% more, position-dependent)
        starting_factor = 1.15
    else:
        # Bench players typically get fewer blocks
        starting_factor = 0.80
    
    # Head-to-head adjustment
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty and not player_df_25.empty:
        h2h = player_df_25[player_df_25['OPP_ABBREVIATION'] == opp_team]
    h2h_factor = 1.0
    if not h2h.empty:
        h2h_avg = h2h['BLK'].mean()
        h2h_factor = h2h_avg / baseline_mean if baseline_mean > 0 else 1.0
        h2h_factor = max(0.80, min(1.20, h2h_factor))  # Wider range for rare events
    
    # Prior mean: baseline adjusted by contextual factors
    prior_mean = baseline_mean * pace_factor * home_factor * rest_factor * h2h_factor * starting_factor
    
    # Prior strength: for rare events like blocks, use more conservative prior
    # Blocks are more volatile, so we give less weight to prior
    prior_strength = min(baseline_games, 30)  # Cap lower for blocks
    prior_strength = max(prior_strength, 3)   # Minimum 3 games
    
    # Gamma prior parameters
    prior_beta = prior_strength / 8.0  # Lower beta = weaker prior (more weight to data)
    prior_alpha = prior_mean * prior_beta
    
    # Ensure valid Gamma distribution
    if prior_alpha < 0.1:  # Lower minimum for rare events
        prior_alpha = 0.1
        prior_beta = prior_alpha / prior_mean
    
    # Get recent observations for likelihood (last 7 games, or more for rare events)
    recent_games = min(10, len(player_df))  # Use more games for blocks (rare events)
    recent_blk = player_df['BLK'].tail(recent_games).values
    
    # Bayesian update: Gamma is conjugate to Poisson
    # Posterior: Gamma(alpha_0 + sum(x_i), beta_0 + n)
    posterior_alpha = prior_alpha + recent_blk.sum()
    posterior_beta = prior_beta + recent_games
    
    # Posterior mean (adjusted lambda)
    lambda_adjusted = posterior_alpha / posterior_beta
    
    # Ensure lambda is positive
    if lambda_adjusted <= 0:
        lambda_adjusted = 0.1
    
    # Posterior variance for uncertainty
    posterior_variance = posterior_alpha / (posterior_beta ** 2)
    posterior_std = np.sqrt(posterior_variance)
    
    return lambda_adjusted, posterior_std

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_blk = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_blocks(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_blk % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_blk) + 1)
    elif target_blk % 1 == 0:
        prob_over_poisson = poisson.sf(target_blk, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_blk), int(target_blk) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_blk,
        'L-5': round(count_line_hits(player_df, target_blk, 'player_blocks', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_blk, 'player_blocks', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_blk, 'player_blocks', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
blocks_df = pd.DataFrame(res)
blocks_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS/player_blocks.csv', index=False)
blocks_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%,IMPLIED_ODDS
0,Evan Mobley,1.5,0.6,0.4,0.47,0.465,0.535,2.152
1,Jakob Poeltl,0.5,0.4,0.5,0.40,0.448,0.552,2.234
2,Noah Clowney,0.5,0.8,0.6,0.40,0.654,0.346,1.529
3,Matas Buzelis,1.5,1.0,0.7,0.47,0.485,0.515,2.063
4,Josh Giddey,0.5,0.6,0.4,0.40,0.401,0.599,2.495
5,Zion Williamson,0.5,0.4,0.2,0.13,0.259,0.741,3.867
6,Kevin Huerter,0.5,0.6,0.7,0.67,0.533,0.467,1.877
7,Donovan Clingan,1.5,0.8,0.7,0.60,0.515,0.485,1.943
8,Kyle Kuzma,0.5,0.4,0.6,0.47,0.456,0.544,2.193
9,Amen Thompson,0.5,0.4,0.5,0.40,0.402,0.598,2.489


# STEALS

In [8]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_steals')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

def compute_bayesian_lambda_steals(player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
                                   team_stats, league_avg_pace, league_avg_tov,
                                   home_flag, current_date, player_name, projectedStartingFive):
    """
    Compute Bayesian posterior lambda for steals using Poisson model.
    
    Uses Gamma prior (conjugate to Poisson) with steals-specific contextual factors.
    Steals are rare events, so we use a more conservative prior similar to blocks.
    """
    eplison = 1e-6
    
    # Get baseline from season data
    if len(player_df) < 5:
        baseline_mean = player_df_25['STL'].mean() if not player_df_25.empty else 0.8
        baseline_games = len(player_df_25)
    else:
        baseline_mean = player_df['STL'].mean()
        baseline_games = len(player_df)
    
    if baseline_mean <= 0:
        baseline_mean = 0.1  # Minimum for steals (rare events)
    
    # Get team and opponent stats
    team_pace = team_stats.at[player_team, 'PACE']
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    opp_tov = team_stats.at[opp_team_id, 'TM_TOV_PCT']
    
    # Contextual adjustments for prior mean (steals-specific)
    # Opponent turnover rate (higher TOV% = more steal opportunities)
    opp_tov_factor = opp_tov / league_avg_tov
    
    # Pace adjustment (more possessions = more steal opportunities)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = expected_pace / league_avg_pace
    
    # Home court advantage (minimal effect for steals)
    home_factor = 1.00  # No home/away effect for steals
    
    # Days rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days
    if days_rested == 1:
        rest_factor = 0.93
    elif days_rested == 2:
        rest_factor = 1.00
    elif 3 <= days_rested <= 5:
        rest_factor = 1.02
    else:
        rest_factor = 0.98
    
    # Starting status adjustment
    # Starting players (especially guards) typically get more steals due to more minutes and better defensive positioning
    is_starting = player_name in projectedStartingFive.get(player_team_abbr, [])
    if is_starting:
        # Starting players get more steals (typically 10-15% more, position-dependent)
        starting_factor = 1.12
    else:
        # Bench players typically get fewer steals
        starting_factor = 0.82
    
    # Head-to-head adjustment
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty and not player_df_25.empty:
        h2h = player_df_25[player_df_25['OPP_ABBREVIATION'] == opp_team]
    h2h_factor = 1.0
    if not h2h.empty:
        h2h_avg = h2h['STL'].mean()
        h2h_factor = h2h_avg / baseline_mean if baseline_mean > 0 else 1.0
        h2h_factor = max(0.80, min(1.20, h2h_factor))  # Wider range for rare events
    
    # Prior mean: baseline adjusted by contextual factors
    prior_mean = baseline_mean * opp_tov_factor * pace_factor * home_factor * rest_factor * h2h_factor * starting_factor
    
    # Prior strength: for rare events like steals, use more conservative prior
    prior_strength = min(baseline_games, 30)  # Cap lower for rare events
    prior_strength = max(prior_strength, 3)   # Minimum 3 games
    
    # Gamma prior parameters
    prior_beta = prior_strength / 8.0  # Lower beta = weaker prior (more weight to data)
    prior_alpha = prior_mean * prior_beta
    
    # Ensure valid Gamma distribution
    if prior_alpha < 0.1:  # Lower minimum for rare events
        prior_alpha = 0.1
        prior_beta = prior_alpha / prior_mean
    
    # Get recent observations for likelihood (last 7-10 games for rare events)
    recent_games = min(10, len(player_df))  # Use more games for steals (rare events)
    recent_stl = player_df['STL'].tail(recent_games).values
    
    # Bayesian update: Gamma is conjugate to Poisson
    # Posterior: Gamma(alpha_0 + sum(x_i), beta_0 + n)
    posterior_alpha = prior_alpha + recent_stl.sum()
    posterior_beta = prior_beta + recent_games
    
    # Posterior mean (adjusted lambda)
    lambda_adjusted = posterior_alpha / posterior_beta
    
    # Ensure lambda is positive
    if lambda_adjusted <= 0:
        lambda_adjusted = 0.1
    
    # Posterior variance for uncertainty
    posterior_variance = posterior_alpha / (posterior_beta ** 2)
    posterior_std = np.sqrt(posterior_variance)
    
    return lambda_adjusted, posterior_std

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_stl = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_steals(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_stl % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_stl) + 1)
    elif target_stl % 1 == 0:
        prob_over_poisson = poisson.sf(target_stl, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_stl), int(target_stl) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_stl,
        'L-5': round(count_line_hits(player_df, target_stl, 'player_steals', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_stl, 'player_steals', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_stl, 'player_steals', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
steals_df = pd.DataFrame(res)
steals_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS/player_steals.csv', index=False)
steals_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Isaiah Jackson,0.5,0.4,0.6,0.40,0.676,0.324
1,Tobias Harris,0.5,0.4,0.4,0.27,0.520,0.480
2,Jordan Clarkson,0.5,0.4,0.4,0.27,0.378,0.622
3,Terance Mann,0.5,0.0,0.2,0.40,0.285,0.715
4,Ziaire Williams,0.5,0.4,0.3,0.47,0.406,0.594
5,Daniel Gafford,0.5,0.6,0.6,0.47,0.586,0.414
6,Max Christie,0.5,0.6,0.5,0.40,0.542,0.458
7,D'Angelo Russell,0.5,0.6,0.4,0.40,0.396,0.604
8,Bruce Brown,0.5,0.4,0.5,0.47,0.637,0.363
9,Cedric Coward,0.5,0.0,0.2,0.27,0.367,0.633


In [9]:
## COMBO PROPS - All Categories

combo_categories = [
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
    'player_turnovers',
    'player_blocks_steals'
]

for category in combo_categories:
    print(f"\nProcessing {category}...")
    
    dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == category)]
    
    if dfs_data.empty:
        print(f"No data found for {category}")
        continue
    
    res = []
    PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))
    
    for _, row in PLAYERS.iterrows():
        PLAYER = row['NAME']
        target_line = row['LINE']

        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        if player_df.empty:
            continue

        res.append({
            'NAME': PLAYER,
            'LINE': target_line,
            'L-5': count_line_hits(player_df, target_line, category, [5])['L-5'],
            'L-10': count_line_hits(player_df, target_line, category, [10])['L-10'],
            'L-15': count_line_hits(player_df, target_line, category, [15])['L-15'],
        })
    
    if res:
        combo_df = pd.DataFrame(res).sort_values(by='L-5', ascending=False).reset_index(drop=True)
        combo_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS/{category}.csv', index=False)
        print(f"Saved {len(combo_df)} players for {category}")
        display(combo_df)
    else:
        print(f"No results for {category}")


Processing player_points_rebounds_assists...
Saved 101 players for player_points_rebounds_assists


,NAME,LINE,L-5,L-10,L-15
0,Keyonte George,30.5,1.0,0.7,0.73
1,Dillon Brooks,25.5,0.8,0.5,0.40
2,Kyle Filipowski,16.0,0.8,0.6,0.53
3,Davion Mitchell,20.5,0.8,0.7,0.60
4,Mikal Bridges,24.5,0.8,0.5,0.60
...,...,...,...,...,...
96,Mike Conley,10.5,0.2,0.4,0.53
97,Kentavious Caldwell-Pope,13.5,0.0,0.2,0.40
98,Jaden Ivey,14.5,0.0,0.0,0.00
99,Scottie Barnes,33.0,0.0,0.1,0.27



Processing player_points_rebounds...
Saved 106 players for player_points_rebounds


,NAME,LINE,L-5,L-10,L-15
0,Duncan Robinson,13.5,1.0,0.8,0.73
1,Bennedict Mathurin,27.5,1.0,0.5,0.33
2,Ayo Dosunmu,17.5,0.8,0.6,0.53
3,Josh Giddey,28.5,0.8,0.7,0.60
4,Tobias Harris,16.5,0.8,0.6,0.40
...,...,...,...,...,...
101,Matas Buzelis,19.5,0.2,0.4,0.47
102,Nick Richards,15.5,0.0,0.0,0.07
103,Scottie Barnes,28.5,0.0,0.1,0.27
104,Kentavious Caldwell-Pope,10.5,0.0,0.2,0.40



Processing player_points_assists...
Saved 97 players for player_points_assists


...

,NAME,LINE,L-5,L-10,L-15
0,Josh Giddey,17.5,1.0,0.9,0.73
1,Donovan Clingan,12.5,0.8,0.4,0.47
2,Lauri Markkanen,8.0,0.8,0.6,0.53
3,Cole Anthony,7.5,0.8,0.7,0.67
4,Lonzo Ball,9.5,0.8,0.5,0.53
...,...,...,...,...,...
56,Sidy Cissoko,6.5,0.2,0.1,0.07
57,Zach LaVine,6.5,0.2,0.3,0.27
58,Jakob Poeltl,11.0,0.2,0.3,0.20
59,Royce O'Neale,8.0,0.0,0.3,0.33



Processing player_turnovers...
Saved 20 players for player_turnovers


,NAME,LINE,L-5,L-10,L-15
0,Jay Huff,0.5,0.8,0.7,0.60
1,Alperen Sengun,3.5,0.6,0.6,0.47
2,Cedric Coward,1.5,0.6,0.5,0.47
3,Isaiah Collier,2.5,0.6,0.4,0.27
4,Keyonte George,3.5,0.6,0.3,0.47
5,Pascal Siakam,2.5,0.4,0.5,0.40
6,Mike Conley,0.5,0.4,0.5,0.60
7,Jaden McDaniels,1.5,0.4,0.7,0.67
8,Ace Bailey,1.5,0.4,0.5,0.40
9,Immanuel Quickley,1.5,0.4,0.4,0.47



Processing player_blocks_steals...
Saved 16 players for player_blocks_steals


,NAME,LINE,L-5,L-10,L-15
0,Evan Mobley,2.5,0.6,0.4,0.60
1,Pascal Siakam,1.5,0.6,0.5,0.47
2,Josh Hart,1.5,0.6,0.4,0.33
3,Dillon Brooks,1.5,0.6,0.6,0.47
4,Oso Ighodaro,1.5,0.6,0.4,0.27
5,Stephen Curry,1.5,0.6,0.5,0.53
6,Mike Conley,0.5,0.6,0.4,0.47
7,Jakob Poeltl,1.5,0.4,0.4,0.27
8,Brandon Williams,1.5,0.4,0.6,0.53
9,Dru Smith,1.5,0.4,0.4,0.40
